# CNN forward pass: one sliding filter and pooling

Companion to HTB Academy Module 290, Section 19. This tiny **single-channel**, **no-padding**, **stride-one** demonstration uses a fixed edge filter. No CNN is trained and no image file is needed.

Read Fᵣ,꜀ = Σᵤ,ᵥ Kᵤ,ᵥ Xᵣ₊ᵤ,꜀₊ᵥ as **F sub r comma c equals the sum over u and v of K sub u comma v times X sub r plus u comma c plus v**. At each filter position, multiply the local patch by the same kernel weights and add. The code uses the cross-correlation convention common in CNN software.

In [ ]:
image = [[0, 1, 1],
         [0, 1, 1],
         [0, 1, 1]]
kernel = [[-1, 1],
          [-1, 1]]  # Positive response when the right side is brighter.
bias = 0

def feature_map_valid_stride_one(values, weights, bias=0):
    """Slide one kernel with no padding; expose each patch and response."""
    height, width = len(values), len(values[0])
    kernel_height, kernel_width = len(weights), len(weights[0])
    if any(len(row) != width for row in values):
        raise ValueError('Image rows must have equal length')
    if any(len(row) != kernel_width for row in weights):
        raise ValueError('Kernel rows must have equal length')
    if kernel_height > height or kernel_width > width:
        raise ValueError('Kernel must fit within the image')
    output = []
    for r in range(height - kernel_height + 1):
        output_row = []
        for c in range(width - kernel_width + 1):
            terms = [weights[u][v] * values[r + u][c + v]
                     for u in range(kernel_height) for v in range(kernel_width)]
            response = sum(terms) + bias  # F_(r,c) = sum_(u,v) K_(u,v) X_(r+u,c+v) + b
            print(f'position ({r},{c}): products={terms}, response={response}')
            output_row.append(response)
        output.append(output_row)
    return output

feature_map = feature_map_valid_stride_one(image, kernel, bias)
print('feature map:', feature_map)
assert feature_map == [[2, 0], [2, 0]]
assert len(feature_map) == 3 - 2 + 1 and len(feature_map[0]) == 3 - 2 + 1

## Downsample the response grid

Pooling aggregates a local window **after** the filter response. This one 2×2 window contains 2, 0, 2, 0. Its maximum is 2; its average is 1. Max and average pooling keep different information. The resulting one-number tensor could be flattened into a one-number vector for a subsequent dense layer, but this notebook deliberately stops before classification.

In [ ]:
pool_window = [value for row in feature_map for value in row]
max_pooled = max(pool_window)
average_pooled = sum(pool_window) / len(pool_window)
print('window:', pool_window, 'max:', max_pooled, 'average:', average_pooled)
assert pool_window == [2, 0, 2, 0]
assert max_pooled == 2 and average_pooled == 1
print('flattened max-pooled vector:', [max_pooled])